# Electric field of point charges

*em · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/coulomb_field)

Coulomb's law and superposition on a grid. Broadcasting builds the separation vector from every charge to every grid point in one go, and a single einsum over the charge index adds up their fields. One charge travels round a circle while the field lines follow it.

**Vector form**

$$\mathbf E(\mathbf x) = \sum_q k\,Q_q\,\frac{\mathbf x - \mathbf x_q}{\lVert\mathbf x - \mathbf x_q\rVert^3}$$

**Index (tensor) form**

$$E_{gi} = k\,Q_q\,\frac{r_{gqi}}{\big(r_{gqj}\,r_{gqj}\big)^{3/2}},\qquad r_{gqi} = x_{gi} - x_{qi}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{gi}$ | `points` | (G, 2) | grid point $g$, component $i \in \{x, y\}$ |
| $x_{qi}$ | `charge_pos` | (Q, 2) | position of charge $q$ |
| $Q_q$ | `charges` | (Q,) | value of each charge |
| $r_{gqi}$ | `points[:, None, :] - charge_pos[None, :, :]` | (G, Q, 2) | broadcasting: the vector from every charge to every grid point |
| $\sqrt{r_{gqj}r_{gqj}}$ | `sqrt(einsum('gqj,gqj->gq', r, r))` | (G, Q) | repeated $j$ ⇒ summed: distances |
| $E_{gi}$ | `einsum('q,gqi,gq->gi', Q, r, 1/d**3)` | (G, 2) | repeated $q$ ⇒ summed: superposition of all charges |

Units with $k = 1$. The charges are point charges in 3D, so each field falls off as $1/r^2$; we only look at the plane that contains them. The arrows point away from positive charges (red) and towards negative ones (blue); with two equal signs a neutral point appears between them where the field vanishes.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/coulomb_field/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    _, points = tensors.grid(params["n_grid"], dev)  # (G, 2) arrows
    line = tensors.x_axis(300, dev)  # (300, 2) points along y = 0
    charges = tensors.charges(params["q_fixed"], params["q_moving"], dev)
    path = tensors.charge_path(72, dev)  # (T, 2, 2)

    E = solve.solve(points, path, charges)  # (T, G, 2)
    E_line = solve.solve(line, path, charges)  # (T, 300, 2)
    E_parts = solve.each_charge(line, path, charges)  # (T, 2, 300, 2)

    print(f"{len(points)} grid points x {len(charges)} charges x {len(path)} positions")
    print(f"superposition check: max |E - sum of parts| = {(E_line - E_parts.sum(1)).abs().max():.2e}")
    return plot.animate(points, E, line, E_line, E_parts, path, charges, out_dir / "coulomb_field.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/coulomb_field/equation.py
r"""Physics → tensor translation.

Coulomb's law plus superposition: the field at a point x is the sum of the fields of all charges q,

    \mathbf E(\mathbf x) = \sum_q k\, Q_q \, \frac{\mathbf x - \mathbf x_q}{\lVert \mathbf x - \mathbf x_q \rVert^3}

In index notation, for grid points g, charges q and spatial components i:

    r_{gqi} = x_{gi} - x_{qi}                       (separation vectors, no sum)
    E_{gi}  = k \, Q_q \, r_{gqi} / |r_{gq}|^3       (q is repeated ⇒ summed: superposition)

with |r_{gq}| = \sqrt{r_{gqj} r_{gqj}} (j repeated ⇒ summed). Broadcasting builds every
separation at once and one einsum adds up all the charges.
"""

import torch


def separations(points: torch.Tensor, charge_pos: torch.Tensor) -> torch.Tensor:
    """r_{gqi} = x_{gi} − x_{qi}.   points: (G, 2), charge_pos: (Q, 2) → (G, Q, 2)"""
    return points[:, None, :] - charge_pos[None, :, :]  # broadcast: (G, 1, 2) − (1, Q, 2)


def electric_field(points, charge_pos, charges, k: float = 1.0) -> torch.Tensor:
    r"""E_{gi} = k Q_q r_{gqi} / |r_{gq}|³   (sum over q).   charges: (Q,) → (G, 2)"""
    r = separations(points, charge_pos)  # (G, Q, 2)
    distance = torch.sqrt(torch.einsum("gqj,gqj->gq", r, r))  # (G, Q)   |r_gq|
    distance = distance.clamp_min(1e-6)  # a grid point exactly on a charge would divide by zero
    return k * torch.einsum("q,gqi,gq->gi", charges, r, 1.0 / distance**3)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/coulomb_field/tensors.py
"""Build the concrete tensors: field points, the charges and the path of the moving charge."""

import math

import torch

L = 2.0  # the plot shows the square −L ≤ x, y ≤ L


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (coordinates (N,), field points (N·N, 2)).

    The points are listed row by row (y outer, x inner), so reshape(N, N, 2) gives rows = y, columns = x.
    """
    xs = torch.linspace(-L, L, n, device=device)  # (N,)
    Y, X = torch.meshgrid(xs, xs, indexing="ij")  # (N, N) each
    points = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=-1)  # (N·N, 2)
    return xs, points


def x_axis(n: int, device: torch.device) -> torch.Tensor:
    """n points along the x axis (y = 0), shape (n, 2): the field is drawn along this line too."""
    xs = torch.linspace(-L, L, n, device=device)
    return torch.stack([xs, torch.zeros_like(xs)], dim=-1)


def charges(q_fixed: float, q_moving: float, device: torch.device) -> torch.Tensor:
    """Q_q of shape (2,): charge 0 sits still, charge 1 moves."""
    return torch.tensor([q_fixed, q_moving], device=device)


def charge_path(n_steps: int, device: torch.device) -> torch.Tensor:
    """x_{tqi} of shape (T, 2, 2): positions of both charges at each of T time steps.

    Charge 0 stays at (−0.6, 0.03); charge 1 goes once round a circle of radius 1.1 about the origin.
    (The small offset keeps charge 0 just off the x axis, where the field is sampled.)
    """
    angle = torch.linspace(0.0, 2 * math.pi, n_steps, device=device)  # (T,)
    path = torch.zeros(n_steps, 2, 2, device=device)
    path[:, 0, 0] = -0.6  # fixed charge
    path[:, 0, 1] = 0.03
    path[:, 1, 0] = 1.1 * torch.cos(angle)  # moving charge, x
    path[:, 1, 1] = 1.1 * torch.sin(angle)  # moving charge, y
    return path


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/coulomb_field/solve.py
"""Evaluate the superposed field for every position of the moving charge."""

import torch

from .equation import electric_field


def solve(points: torch.Tensor, path: torch.Tensor, charges: torch.Tensor) -> torch.Tensor:
    """Return E of shape (T, G, 2): the field at every point for each of the T charge positions.

    Each time step is one call to electric_field — all G points and both charges at once.
    """
    return torch.stack([electric_field(points, positions, charges) for positions in path])


def each_charge(points: torch.Tensor, path: torch.Tensor, charges: torch.Tensor) -> torch.Tensor:
    """Return the field of every charge on its own, shape (T, Q, G, 2).

    Summing over Q gives back solve(...): that is the superposition principle.
    """
    alone = [solve(points, path[:, q : q + 1], charges[q : q + 1]) for q in range(len(charges))]
    return torch.stack(alone, dim=1)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/coulomb_field/plot.py
"""Animate the field: direction arrows on the grid, and E_x along the x axis split into its two parts."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif

from .tensors import L


def animate(points, E, line, E_line, E_parts, path_q, charges, path: Path) -> Path:
    p = points.cpu().numpy()  # (G, 2)
    field = E.cpu().numpy()  # (T, G, 2)
    strength = np.linalg.norm(field, axis=-1)  # (T, G)
    direction = field / strength[..., None]  # unit arrows: only the direction is drawn
    x_line = line[:, 0].cpu().numpy()  # (n,)
    Ex_total = E_line[..., 0].cpu().numpy()  # (T, n)
    Ex_parts = E_parts[..., 0].cpu().numpy()  # (T, Q, n)
    Ex_total[np.abs(Ex_total) > 6] = np.nan  # don't draw the jump through ±∞ right next to a charge
    Ex_parts[np.abs(Ex_parts) > 6] = np.nan
    pos = path_q.cpu().numpy()  # (T, 2, 2)
    q = charges.cpu().numpy()  # (2,)
    colours = ["#dc2626" if value > 0 else "#2563eb" for value in q]

    fig, (ax, ax_line) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1, 1]})

    # left: arrows coloured by log |E|, and the two charges
    arrows = ax.quiver(
        p[:, 0], p[:, 1], direction[0, :, 0], direction[0, :, 1], np.log10(strength[0]),
        cmap="viridis", clim=(-1.0, 1.5), pivot="mid", scale=30, width=0.006,
    )  # fmt: skip
    dots = ax.scatter(pos[0, :, 0], pos[0, :, 1], c=colours, s=80, edgecolors="k", zorder=3)
    ax.set_xlim(-L, L)
    ax.set_ylim(-L, L)
    ax.set_aspect("equal")
    ax.grid(False)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.axhline(0, color="#9ca3af", lw=0.8, ls=":")
    ax.set_title("direction of E (colour: log₁₀|E|)")

    # right: E_x along the dotted line y = 0 — the total is the sum of the two dashed curves
    part_lines = [
        ax_line.plot([], [], "--", lw=1.2, color=colours[k], label=f"charge {q[k]:+g} alone")[0] for k in range(2)
    ]
    (total_line,) = ax_line.plot([], [], color="k", lw=1.8, label="sum over q")
    ax_line.set_xlim(-L, L)
    ax_line.set_ylim(-3, 3)
    ax_line.axhline(0, color="#6b7280", lw=0.8)
    ax_line.set_xlabel("x  (along y = 0)")
    ax_line.set_ylabel("E_x")
    ax_line.set_title("superposition")
    ax_line.legend(loc="upper right", fontsize=8)

    def update(step: int):
        arrows.set_UVC(direction[step, :, 0], direction[step, :, 1], np.log10(strength[step]))
        dots.set_offsets(pos[step])
        for k, curve in enumerate(part_lines):
            curve.set_data(x_line, Ex_parts[step, k])
        total_line.set_data(x_line, Ex_total[step])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(pos), 72), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'q_fixed': 1.0,
    'q_moving': -1.0,
    'n_grid': 21
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.coulomb_field")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.coulomb_field.main")
problem_dir = Path("problems/em/coulomb_field")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
